In [30]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

import sys
sys.path.append('../../05_src/')

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [31]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0.7,
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

In [32]:
from langchain.tools import tool
import requests
import json

## Tool for accessing open-source API tool. In this case, it is space travel news
@tool
def get_space_news(n:int=1):
    """
    Returns news about space travel from SpaceflightNews API.
    """
    url = "https://spaceflightnewsapi.net/"
    params = {
        "count": n
    }
    response = requests.get(url, params=params)
    resp_dict = json.loads(response.text)
    facts_list = resp_dict.get("data", [])
    facts = "\n".join([f"{i+1}. {fact}\n" for i, fact in enumerate(facts_list)])
    return facts

## Tool for getting embeddings. I used the small text embedding model introduced in class.
## will update for this to get something useful instead of just getting embedding
@tool
def get_embedding(text, model="text-embedding-3-small"):
    """
    semantic embedding using text-embedding-3-small model
    """
    text = text.replace("\n", " ")
    return client.embeddings.create(input=text, model=model).data[0].embedding



def simple_function_gravity(mass:float,altitude:float):
    """
    simple function that gets force of gravity earth exerts on you at sea level given your mass and altitude
    """
    F_grav = mass*5.98*10**24*6.67*10**-11/(6.38*10**6+altitude)**2
    F_grav = str(round(F_grav))
    output = f"{F_grav} Newtons is the force of gravity earth exerts on you at sea level"
    return output
    
    
tools = [get_space_news,get_embedding,simple_function_gravity]
instructions = ['placeholder, will make txt file']

In [33]:
from langgraph.graph import StateGraph, MessagesState, START
from langchain.chat_models import init_chat_model
from langgraph.prebuilt.tool_node import ToolNode, tools_condition
from langchain_core.messages import SystemMessage,  HumanMessage

from dotenv import load_dotenv
import json
import requests
import os

def call_model(state: MessagesState):
    """LLM decides whether to call a tool or not"""
    response = chat_agent.bind_tools(tools).invoke( [SystemMessage(content=instructions)] + state["messages"])
    return {
        "messages": [response]
    }

def get_graph():
    
    builder = StateGraph(MessagesState)
    builder.add_node(call_model)
    builder.add_node(ToolNode(tools))
    builder.add_edge(START, "call_model")
    builder.add_conditional_edges(
        "call_model",
        tools_condition,
    )
    builder.add_edge("tools", "call_model")
    graph = builder.compile()
    return graph

In [34]:

from langchain_core.messages import HumanMessage, AIMessage
import gradio as gr
from dotenv import load_dotenv
import os

from utils.logger import get_logger

_logs = get_logger(__name__)

llm = get_graph()

load_dotenv('.secrets')

def course_chat(message: str, history: list[dict]) -> str:
    langchain_messages = []
    n = 0
    _logs.debug(f"History: {history}")
    for msg in history:
        if msg['role'] == 'user':
            langchain_messages.append(HumanMessage(content=msg['content']))
        elif msg['role'] == 'assistant':
            langchain_messages.append(AIMessage(content=msg['content']))
            n += 1
    langchain_messages.append(HumanMessage(content=message))

    state = {
        "messages": langchain_messages,
        "llm_calls": n
    }

    response = llm.invoke(state)
    return response['messages'][len(response['messages']) - 1].content

chat = gr.ChatInterface(
    fn=course_chat,
    type="messages"
)

if __name__ == "__main__":
    _logs.info('Starting Course Chat App...')
    chat.launch()

2026-02-27 20:52:07,982, 3854977320.py, 40, INFO, Starting Course Chat App...


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/exx/Desktop/dsi/asg7/deploying-ai/deploying-ai-env/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exx/Desktop/dsi/asg7/deploying-ai/deploying-ai-env/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exx/Desktop/dsi/asg7/deploying-ai/deploying-ai-env/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exx/Desktop/dsi/asg7/deploying-ai/deploying-ai-env/lib/python3.12/site-packages/gradio/blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/exx/Desktop/ds